# CrewAI Literature Review Crew

**Project:** Multi-Agent Academic Literature Review Pipeline  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent academic research crew** that automates the literature review process — from discovering relevant papers to identifying research gaps and generating a formatted bibliography.

### The Agent Roster

| # | Agent | Role | Deliverable |
|---|-------|------|------------|
| 1 | **Literature Searcher** | Discovery Specialist | Curated list of relevant papers with abstracts and key findings |
| 2 | **Summarizer** | Synthesis Expert | Thematic summaries organized by research sub-topics |
| 3 | **Gap Finder** | Critical Analyst | Map of what's been studied vs. what's missing or underexplored |
| 4 | **Bibliography Agent** | Citation Formatter | Properly formatted reference list with annotations |

### Multi-Agent Research Flow

```
Research Topic
     │
     ▼
┌────────────────────┐
│ Literature Searcher │  "WHAT exists?" — discovers papers, extracts metadata
└─────────┬──────────┘
          │ Paper catalog (titles, authors, years, key findings)
          ▼
┌────────────────────┐
│    Summarizer      │  "WHAT does it all mean?" — synthesizes themes across papers
└─────────┬──────────┘
          │ Thematic synthesis organized by sub-topics
          ▼
┌────────────────────┐
│    Gap Finder      │  "WHAT'S missing?" — identifies holes, contradictions, future directions
└─────────┬──────────┘
          │ Gap analysis with research opportunities
          ▼
┌────────────────────┐
│ Bibliography Agent │  "HOW to cite it?" — formats citations, adds annotations
└────────────────────┘
          │
          ▼
  Complete literature review package
```

### Why Multi-Agent for Literature Reviews?

A single monolithic prompt asking an LLM to "do a literature review" produces shallow, generic output. Breaking it into four specialized stages produces better results because:

1. **Search and synthesis are different skills** — Finding relevant papers requires breadth; summarizing them requires depth
2. **Gap analysis needs the full picture** — The gap finder can only identify what's missing after seeing what exists
3. **Citation formatting is mechanical** — Separating it from analysis prevents format concerns from diluting intellectual work
4. **Each agent builds on accumulated context** — The bibliography agent sees the paper catalog, the synthesis, AND the gaps

This mirrors how real research teams work: a research assistant gathers sources, a domain expert synthesizes findings, a senior researcher identifies gaps, and a librarian handles citations.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

All agents use the same local Ollama model. The different outputs come from role design, not model selection.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### Multi-Agent Design Philosophy for Research

In a literature review crew, each agent mirrors a stage of the academic review process:

| Research Stage | CrewAI Agent | Cognitive Mode |
|---------------|-------------|---------------|
| Discovery | Literature Searcher | **Divergent** — cast a wide net, find everything relevant |
| Synthesis | Summarizer | **Convergent** — distill many papers into coherent themes |
| Critique | Gap Finder | **Analytical** — compare what exists vs. what's needed |
| Documentation | Bibliography Agent | **Systematic** — format, organize, annotate precisely |

**Why this decomposition works:** Each stage requires a fundamentally different cognitive approach. Mixing them in one agent forces the LLM to switch modes constantly, degrading output quality.

### 3.1 Literature Searcher Agent

**Research flow role:** The **discovery engine** — casts a wide net to find relevant papers, extracts structured metadata, and creates the raw material that all downstream agents work with.

**Critical design choice:** The backstory emphasizes systematic search strategies (keyword expansion, citation chaining, cross-disciplinary scanning) rather than just "find papers." This produces more comprehensive and diverse results.

In [ ]:
literature_searcher = Agent(
    role="Academic Literature Discovery Specialist",
    goal=(
        "Conduct a comprehensive literature search on the given research topic. "
        "Identify 12-15 highly relevant papers spanning foundational work, recent "
        "advances, and contrarian perspectives. For each paper, extract title, "
        "authors, year, venue, key findings, methodology, and relevance score."
    ),
    backstory=(
        "You are a research librarian and literature discovery specialist with 12 years "
        "of experience supporting PhD students and research labs at top universities. "
        "You use systematic search strategies: (1) keyword expansion — identifying synonyms, "
        "related terms, and sub-field terminology, (2) citation chaining — following references "
        "forward and backward from seminal papers, (3) cross-disciplinary scanning — checking "
        "adjacent fields that study the same problem differently. You always organize results "
        "by relevance tier (foundational, recent, contrarian) and flag which papers are most "
        "cited, which are newest, and which challenge conventional wisdom. Your output is a "
        "structured paper catalog with consistent metadata for each entry."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {literature_searcher.role}")

### 3.2 Summarizer Agent

**Research flow role:** The **synthesis engine** — transforms a collection of individual paper summaries into a coherent thematic narrative. This is the hardest intellectual task in a literature review.

**Design insight:** The backstory explicitly mentions "thematic synthesis" and "not paper-by-paper" because LLMs default to summarizing each paper individually. Forcing a thematic structure produces dramatically better output.

In [ ]:
summarizer = Agent(
    role="Research Synthesis & Thematic Analysis Expert",
    goal=(
        "Synthesize the discovered papers into a coherent thematic literature review. "
        "Organize findings by research themes (NOT paper-by-paper), identify consensus "
        "positions, highlight methodological trends, and note areas of disagreement "
        "across the literature."
    ),
    backstory=(
        "You are a senior research fellow who has written and reviewed literature reviews "
        "for top-tier journals (Nature Reviews, Annual Review of Computer Science, ACM "
        "Computing Surveys). You know that the difference between a weak and strong "
        "literature review is organization: weak reviews summarize papers one by one; "
        "strong reviews identify 4-6 themes and weave papers together under each theme. "
        "You follow the thematic synthesis method: (1) identify recurring themes across "
        "papers, (2) group papers under each theme, (3) compare findings within themes — "
        "where do papers agree, disagree, or complement each other? (4) track methodological "
        "evolution — how have approaches changed over time? Your summaries always cite "
        "specific papers by name and distinguish between established consensus vs. emerging "
        "or contested findings."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {summarizer.role}")

### 3.3 Gap Finder Agent

**Research flow role:** The **critical lens** — this agent's entire purpose is to identify what the literature does NOT cover. This is the highest-value contribution of a literature review because it directly informs future research directions.

**Why a separate agent?** When summarizers also do gap analysis, they tend to be overly generous — noting only minor gaps to avoid contradicting their own synthesis. A dedicated critic agent has no stake in the summary's completeness, producing more honest gap identification.

In [ ]:
gap_finder = Agent(
    role="Research Gap Analyst & Future Directions Strategist",
    goal=(
        "Critically analyze the literature synthesis to identify research gaps, "
        "methodological blind spots, understudied populations or contexts, "
        "contradictions that remain unresolved, and promising future research "
        "directions. Rank gaps by significance and tractability."
    ),
    backstory=(
        "You are a research strategy consultant who advises PhD students and research "
        "groups on where to focus their work for maximum impact. You have reviewed 500+ "
        "literature reviews and your specialty is finding what everyone else missed. You "
        "analyze gaps across five dimensions: (1) Topical gaps — sub-topics that should "
        "exist but don't, (2) Methodological gaps — approaches not yet tried or validated, "
        "(3) Population gaps — groups, contexts, or domains not studied, (4) Temporal gaps — "
        "how old is the evidence? Has the field moved on? (5) Integration gaps — findings "
        "from adjacent fields that haven't been connected. For each gap, you assess: how "
        "significant is it? How tractable is it to address? What would a study look like? "
        "You are constructively critical — you don't just point out problems, you suggest "
        "actionable research directions."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {gap_finder.role}")

### 3.4 Bibliography Agent

**Research flow role:** The **documentation engine** — transforms the informal paper references throughout the review into a properly formatted, annotated bibliography. This agent sees ALL prior outputs and creates a unified reference section.

**Design choice:** The backstory specifies APA 7th edition format and includes annotation requirements. This prevents the common LLM behavior of generating inconsistent or partially formatted citations.

In [ ]:
bibliography_agent = Agent(
    role="Academic Citation & Bibliography Specialist",
    goal=(
        "Create a properly formatted annotated bibliography from all papers "
        "referenced in the literature review. Each entry should include a "
        "standard citation (APA 7th edition), a 2-3 sentence annotation "
        "describing the paper's contribution, and a relevance tag indicating "
        "how it connects to the research topic."
    ),
    backstory=(
        "You are an academic librarian and citation specialist who manages "
        "reference databases for a large research university. You have formatted "
        "10,000+ citations across APA, MLA, Chicago, and IEEE styles. You are "
        "meticulous about citation accuracy: author order, year placement, journal "
        "italicization, DOI formatting, and page numbers. For this project you use "
        "APA 7th edition exclusively. Your annotated bibliographies add value beyond "
        "formatting: each annotation summarizes the paper's key contribution in 2-3 "
        "sentences and tags its role in the literature (foundational, methodology, "
        "empirical evidence, contrarian view, review article). You organize the "
        "bibliography alphabetically by first author and include a thematic index "
        "cross-referencing each paper to the review themes identified by the summarizer."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {bibliography_agent.role}")

### Agent Roster Summary

In [ ]:
agents = [literature_searcher, summarizer, gap_finder, bibliography_agent]

print("=" * 70)
print("LITERATURE REVIEW CREW ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Goal: {agent.goal[:90]}...")
    print(f"    Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)

## 4. Define Tasks & Research Flow

### Understanding the Multi-Agent Research Flow

The research flow in this crew has a **build-criticize-document** pattern:

| Phase | Agent | Cognitive Task | Input | Output |
|-------|-------|---------------|-------|--------|
| **Build** (Discovery) | Searcher | Find raw material | Topic | Paper catalog |
| **Build** (Synthesis) | Summarizer | Organize and connect | Paper catalog | Thematic review |
| **Criticize** | Gap Finder | Challenge completeness | Thematic review | Gap analysis |
| **Document** | Bibliography | Formalize references | All prior outputs | Annotated bibliography |

**Key insight:** The flow alternates between constructive work (search, synthesize) and critical work (find gaps). This mirrors the academic peer review process where reviewers specifically look for what's missing.

**Context accumulation:** Each agent sees ALL previous outputs:
- Summarizer sees: paper catalog
- Gap Finder sees: paper catalog + thematic synthesis
- Bibliography Agent sees: paper catalog + synthesis + gap analysis

### 4.1 Configure the Research Topic

The pipeline is parameterized — change the topic and re-run to produce a literature review on any subject.

In [ ]:
# =====================================================
# CONFIGURE YOUR RESEARCH TOPIC HERE
# =====================================================
RESEARCH_TOPIC = "Retrieval-Augmented Generation (RAG) for Knowledge-Intensive NLP Tasks"
RESEARCH_SCOPE = "Focus on architectures, retrieval mechanisms, evaluation methods, and real-world applications of RAG systems from 2020 to present"
DISCIPLINE = "Natural Language Processing / Information Retrieval"
TARGET_AUDIENCE = "ML researchers and practitioners building production RAG systems"
KEY_TERMS = ["retrieval-augmented generation", "RAG", "dense retrieval", "knowledge grounding", "hallucination reduction", "vector databases"]
# =====================================================

print(f"Topic: {RESEARCH_TOPIC}")
print(f"Scope: {RESEARCH_SCOPE}")
print(f"Discipline: {DISCIPLINE}")
print(f"Key Terms: {', '.join(KEY_TERMS)}")

### 4.2 Task 1 — Literature Search

**Research flow:** Discovery phase — cast a wide net to find all relevant work.

**Handoff:** The paper catalog becomes the raw material for thematic synthesis.

In [ ]:
task_search = Task(
    description=(
        f"Conduct a comprehensive literature search on:\n"
        f"TOPIC: {RESEARCH_TOPIC}\n"
        f"SCOPE: {RESEARCH_SCOPE}\n"
        f"DISCIPLINE: {DISCIPLINE}\n"
        f"KEY TERMS: {', '.join(KEY_TERMS)}\n\n"
        "Your search MUST produce a structured paper catalog covering:\n"
        "1. **Foundational Papers** (3-4): The seminal works that defined this research area\n"
        "2. **Core Recent Papers** (5-6): The most significant recent advances (2022-2024)\n"
        "3. **Methodological Papers** (2-3): Papers that introduced key techniques or evaluation methods\n"
        "4. **Contrarian/Critical Papers** (2-3): Papers that challenge mainstream approaches\n\n"
        "For EACH paper, provide:\n"
        "- Title, Authors (first author et al.), Year, Venue\n"
        "- Key finding in 1-2 sentences\n"
        "- Methodology used\n"
        "- Relevance tier: Foundational / Core Recent / Methodological / Contrarian\n"
        "- Relevance score (1-10) to the research topic\n"
    ),
    expected_output=(
        "A structured paper catalog with 12-15 papers organized by relevance tier "
        "(Foundational, Core Recent, Methodological, Contrarian). Each entry includes "
        "title, authors, year, venue, key finding, methodology, and relevance score."
    ),
    agent=literature_searcher,
)

print(f"Task created: Literature Search -> {task_search.agent.role}")

### 4.3 Task 2 — Thematic Synthesis

**Research flow:** Synthesis phase — transform individual papers into a coherent narrative.

**Handoff:** Receives the paper catalog; produces an organized thematic review that the gap finder will scrutinize.

In [ ]:
task_synthesize = Task(
    description=(
        "Using the paper catalog from the literature search, synthesize a "
        "thematic literature review.\n\n"
        "Requirements:\n"
        "1. **Identify 4-6 Themes**: Group papers into coherent research themes "
        "   (NOT paper-by-paper summaries)\n"
        "2. **Within Each Theme**:\n"
        "   - Summarize the main findings across papers\n"
        "   - Note where papers agree (consensus)\n"
        "   - Note where papers disagree (controversy)\n"
        "   - Track methodological evolution within the theme\n"
        "3. **Cross-Theme Connections**: Where do themes overlap or interact?\n"
        "4. **Methodological Landscape**: What are the dominant research methods? "
        "   What's the balance between theoretical vs. empirical work?\n"
        "5. **Timeline**: How has the field evolved? What are the generational shifts?\n\n"
        "CRITICAL: Cite specific papers by name within each theme. A synthesis without "
        "citations is just opinion."
    ),
    expected_output=(
        "A thematic literature review with 4-6 identified themes, within-theme "
        "synthesis with consensus/disagreement notes, cross-theme connections, "
        "methodological landscape, and timeline of field evolution. Papers cited by name."
    ),
    agent=summarizer,
)

print(f"Task created: Thematic Synthesis -> {task_synthesize.agent.role}")

### 4.4 Task 3 — Gap Analysis

**Research flow:** Critique phase — challenge the completeness and identify what's missing.

**Handoff:** Receives both the paper catalog AND the thematic synthesis. This dual context lets the gap finder compare what was found vs. what should have been found.

In [ ]:
task_gap_analysis = Task(
    description=(
        "Critically analyze the literature synthesis to identify research gaps "
        "and future directions.\n\n"
        "Analyze gaps across FIVE dimensions:\n"
        "1. **Topical Gaps**: Sub-topics within the research area that have NO or "
        "   INSUFFICIENT coverage in the discovered literature\n"
        "2. **Methodological Gaps**: Research approaches that have NOT been tried, "
        "   validated, or compared for this problem\n"
        "3. **Context/Population Gaps**: Domains, languages, industries, or user groups "
        "   that are underrepresented in the literature\n"
        "4. **Temporal Gaps**: Areas where the evidence is outdated or hasn't been "
        "   re-examined with current methods/data\n"
        "5. **Integration Gaps**: Findings from adjacent fields that haven't been "
        "   connected to this research area\n\n"
        "For EACH identified gap:\n"
        "- Describe the gap clearly\n"
        "- Rate significance: High / Medium / Low\n"
        "- Rate tractability: Easy / Moderate / Hard\n"
        "- Suggest a specific research question that would address it\n"
        "- Estimate the impact of filling this gap\n\n"
        "Also identify:\n"
        "- **Unresolved Contradictions**: Where do studies disagree without resolution?\n"
        "- **Weak Evidence Areas**: Where are claims made with insufficient evidence?\n"
    ),
    expected_output=(
        "A gap analysis report with gaps organized by five dimensions (topical, "
        "methodological, context, temporal, integration). Each gap includes "
        "significance/tractability ratings, a suggested research question, and "
        "impact estimate. Plus unresolved contradictions and weak evidence areas."
    ),
    agent=gap_finder,
)

print(f"Task created: Gap Analysis -> {task_gap_analysis.agent.role}")

### 4.5 Task 4 — Annotated Bibliography

**Research flow:** Documentation phase — formalize all references into a usable bibliography.

**Handoff:** Receives ALL prior outputs (paper catalog + synthesis + gaps). Uses the full context to create annotations that reflect how each paper fits into the overall research landscape.

In [ ]:
task_bibliography = Task(
    description=(
        "Create an annotated bibliography for all papers discussed in the "
        "literature review.\n\n"
        "Requirements:\n"
        "1. **Citation Format**: APA 7th edition for every entry\n"
        "2. **Annotation** (2-3 sentences per paper):\n"
        "   - What is the paper's key contribution?\n"
        "   - How does it connect to the themes identified in the synthesis?\n"
        "   - What gaps (from the gap analysis) does it partially address or leave open?\n"
        "3. **Relevance Tag** for each paper:\n"
        "   - Foundational / Empirical / Methodological / Review / Contrarian\n"
        "4. **Thematic Index**: A cross-reference table mapping each paper to the "
        "   review themes it appears under\n"
        "5. **Reading Priority**: Rank papers into Must-Read (top 5), Recommended, "
        "   and Supplementary categories\n"
    ),
    expected_output=(
        "An annotated bibliography with APA 7th edition citations, 2-3 sentence "
        "annotations per paper, relevance tags, a thematic cross-reference index, "
        "and a reading priority ranking."
    ),
    agent=bibliography_agent,
)

print(f"Task created: Annotated Bibliography -> {task_bibliography.agent.role}")

### Research Flow Visualization

In [ ]:
tasks = [task_search, task_synthesize, task_gap_analysis, task_bibliography]
task_names = ["Literature Search", "Thematic Synthesis", "Gap Analysis", "Annotated Bibliography"]
flow_labels = [
    "DISCOVER — paper catalog with metadata",
    "SYNTHESIZE — thematic review with citations",
    "CRITIQUE — gap map with research questions",
    "DOCUMENT — formatted bibliography with annotations",
]

print("MULTI-AGENT RESEARCH FLOW")
print("=" * 70)
for i, (name, task, flow) in enumerate(zip(task_names, tasks, flow_labels), 1):
    print(f"  Phase {i}: {name}")
    print(f"    Agent: {task.agent.role}")
    print(f"    Action: {flow}")
    if i < len(tasks):
        print(f"    {'':8}|")
        print(f"    {'':8}| (context accumulates)")
        print(f"    {'':8}|")
print("=" * 70)
print(f"\nPhase 4 agent sees ALL outputs from Phases 1 + 2 + 3")

## 5. Assemble and Run the Crew

### Crew Coordination for Research Tasks

The `Crew` manages the research pipeline with key coordination responsibilities:

| Responsibility | How It Works |
|---------------|-------------|
| **Phase sequencing** | Tasks run in order: Search → Synthesize → Gap → Bibliography |
| **Context accumulation** | Each agent receives ALL previous agents' outputs as context |
| **Agent isolation** | Each agent works independently — no inter-agent communication |
| **Output preservation** | Every agent's deliverable is saved separately for inspection |

**Sequential process** is ideal for literature reviews because each phase genuinely depends on the previous one — you can't synthesize papers you haven't found yet.

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")

In [ ]:
# Execute the literature review pipeline
print("=" * 70)
print(f"LAUNCHING LITERATURE REVIEW CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Topic: {RESEARCH_TOPIC}")
print(f"Flow: Search -> Synthesize -> Gap Analysis -> Bibliography")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — Annotated Bibliography

In [ ]:
print("=" * 70)
print("FINAL OUTPUT — Annotated Bibliography")
print("=" * 70)
print(result.raw)

### 6.2 Individual Phase Outputs

Inspect each phase's deliverable to trace how information flowed through the research pipeline.

In [ ]:
for name, task in zip(task_names, tasks):
    print("\n" + "=" * 70)
    print(f"PHASE OUTPUT: {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2500:
            print(text[:2500])
            print(f"\n... [{len(text) - 2500} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Pipeline Metrics

In [ ]:
print("LITERATURE REVIEW PIPELINE METRICS")
print("=" * 60)
print(f"{'Phase':<25} {'Agent':<35} {'Output':>8}")
print("-" * 60)

total = 0
for name, task in zip(task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    print(f"{name:<25} {task.agent.role.split('&')[0].strip():<35} {length:>6,}")

print("-" * 60)
print(f"{'TOTAL':<60} {total:>6,}")

if hasattr(result, 'token_usage') and result.token_usage:
    print(f"\nTokens: {result.token_usage.get('total_tokens', 'N/A')}")
else:
    print("\nToken usage: not tracked (typical with local Ollama)")

## 7. Save Complete Literature Review

In [ ]:
# Build and save the complete literature review document
review_lines = [
    f"# Literature Review: {RESEARCH_TOPIC}",
    f"**Generated:** {datetime.now().isoformat()}",
    f"**Scope:** {RESEARCH_SCOPE}",
    f"**Discipline:** {DISCIPLINE}",
    f"**Target Audience:** {TARGET_AUDIENCE}",
    "---",
]

for name, task in zip(task_names, tasks):
    review_lines.append(f"\n## {name}\n")
    review_lines.append(task.output.raw if task.output else "[No output]")
    review_lines.append("\n---")

review = "\n".join(review_lines)
with open("literature_review.md", "w", encoding="utf-8") as f:
    f.write(review)

# Save gap analysis separately (often the most actionable deliverable)
if task_gap_analysis.output:
    with open("research_gaps.md", "w", encoding="utf-8") as f:
        f.write(task_gap_analysis.output.raw)

print(f"Files saved:")
print(f"  literature_review.md ({len(review):,} chars)")
if task_gap_analysis.output:
    print(f"  research_gaps.md ({len(task_gap_analysis.output.raw):,} chars)")

## 8. Experiment: Different Research Topic

The entire pipeline is reusable — change the topic and re-run for any research domain.

In [ ]:
# =====================================================
# SECOND RESEARCH TOPIC
# =====================================================
TOPIC_2 = "Explainability and Interpretability of Large Language Models"
SCOPE_2 = "Focus on post-hoc explanation methods, mechanistic interpretability, and evaluation of explanations for LLMs (2021-present)"
DISCIPLINE_2 = "AI Safety / Machine Learning Interpretability"
AUDIENCE_2 = "AI safety researchers, ML practitioners, and policy-makers"
TERMS_2 = ["LLM interpretability", "mechanistic interpretability", "attention visualization", "probing classifiers", "circuit analysis", "explanation faithfulness"]

print(f"Topic 2: {TOPIC_2}")
print(f"Scope: {SCOPE_2}")
print(f"Key Terms: {', '.join(TERMS_2)}")

In [ ]:
# Build task set for the second topic
task2_search = Task(
    description=(
        f"Comprehensive literature search on: {TOPIC_2}\n"
        f"Scope: {SCOPE_2}\n"
        f"Key terms: {', '.join(TERMS_2)}\n\n"
        "Find 12-15 papers across: foundational (3-4), core recent (5-6), "
        "methodological (2-3), contrarian (2-3). Provide title, authors, year, "
        "venue, key finding, methodology, relevance tier, and score."
    ),
    expected_output="Structured paper catalog with 12-15 papers by relevance tier.",
    agent=literature_searcher,
)

task2_synthesize = Task(
    description=(
        "Synthesize the paper catalog into a thematic literature review. "
        "Identify 4-6 themes, note consensus/disagreement within themes, "
        "track methodological evolution. Cite papers by name."
    ),
    expected_output="Thematic synthesis with 4-6 themes and paper citations.",
    agent=summarizer,
)

task2_gaps = Task(
    description=(
        "Analyze gaps across 5 dimensions: topical, methodological, "
        "context/population, temporal, integration. Rate significance "
        "and tractability. Suggest research questions for each gap."
    ),
    expected_output="Gap analysis with rated gaps and suggested research questions.",
    agent=gap_finder,
)

task2_bib = Task(
    description=(
        "Create APA 7th annotated bibliography with 2-3 sentence annotations, "
        "relevance tags, thematic index, and reading priority ranking."
    ),
    expected_output="Annotated bibliography with cross-reference index.",
    agent=bibliography_agent,
)

tasks_2 = [task2_search, task2_synthesize, task2_gaps, task2_bib]
print(f"Second topic tasks created: {len(tasks_2)} tasks")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Topic: {TOPIC_2}")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print(f"BIBLIOGRAPHY — {TOPIC_2[:50]}...")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Reviews

In [ ]:
print("LITERATURE REVIEW COMPARISON")
print("=" * 70)
print(f"{'Phase':<25} {'RAG (chars)':<18} {'LLM Interp (chars)':<18}")
print("-" * 70)

for name, t1, t2 in zip(task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{name:<25} {len1:<18,} {len2:<18,}")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 70)
print(f"{'TOTAL':<25} {total1:<18,} {total2:<18,}")

## 10. Multi-Agent Research Flow — Deep Dive

### Flow Pattern 1: Linear Pipeline (This Notebook)

```
Search -> Synthesize -> Gaps -> Bibliography
```

Each agent builds on all prior outputs. Simple, predictable, and sufficient for standard literature reviews.

**When to use:** Single-topic reviews where the scope is well-defined.

### Flow Pattern 2: Parallel Discovery + Convergent Synthesis

```
Database Searcher ──┐
                    ├──> Synthesizer ──> Gap Finder ──> Bibliography
Citation Chainer ───┘
```

Two searchers work in parallel (one doing keyword search, one doing citation chaining), then a synthesizer merges their findings. This produces more comprehensive discovery.

```python
# CrewAI implementation with explicit context
task_keyword_search = Task(..., agent=keyword_searcher)
task_citation_chain = Task(..., agent=citation_chainer)
task_synthesis = Task(..., agent=summarizer, context=[task_keyword_search, task_citation_chain])
```

### Flow Pattern 3: Iterative Refinement Loop

```
Search -> Synthesize -> Gap Finder ──[gaps found]──> Search (targeted)
                                  ──[sufficient]──> Bibliography
```

The gap finder sends the searcher back with specific requests to fill identified gaps. This requires custom task callbacks in CrewAI.

### Flow Pattern 4: Multi-Perspective Review

```
                    ┌──> ML Perspective Synthesizer  ──┐
Paper Catalog ──────┤                                  ├──> Integrator -> Bibliography
                    └──> Ethics Perspective Synthesizer ┘
```

Multiple synthesizers analyze the same papers from different angles, then an integrator combines their perspectives. Excellent for interdisciplinary reviews.

### Choosing the Right Flow

| Flow Pattern | Best For | Complexity |
|-------------|----------|-----------|
| Linear Pipeline | Well-scoped single-discipline reviews | Low |
| Parallel Discovery | Comprehensive reviews requiring exhaustive coverage | Medium |
| Iterative Refinement | Exploratory reviews where the scope evolves | High |
| Multi-Perspective | Interdisciplinary or policy-oriented reviews | High |

## 11. Advanced: Explicit Context for Parallel Phases

Demonstrate how to give agents selective context access — useful for parallel search strategies that converge into a single synthesis.

In [ ]:
# Parallel discovery demo: keyword search + citation search -> merged synthesis

keyword_search_task = Task(
    description=(
        f"Keyword-based literature search for: {RESEARCH_TOPIC}\n"
        f"Use these search terms: {', '.join(KEY_TERMS)}\n"
        "Find 8-10 papers through keyword matching. Focus on titles and abstracts."
    ),
    expected_output="Paper catalog from keyword search (8-10 papers).",
    agent=literature_searcher,
)

citation_search_task = Task(
    description=(
        f"Citation-chain search for: {RESEARCH_TOPIC}\n"
        "Start from the seminal RAG paper (Lewis et al., 2020) and follow "
        "backward/forward citations to find 5-7 papers that keyword search "
        "might miss. Focus on methodological lineage."
    ),
    expected_output="Paper catalog from citation chaining (5-7 papers).",
    agent=literature_searcher,
)

# Synthesizer sees BOTH search results
merged_synthesis_task = Task(
    description=(
        "Merge and synthesize papers from BOTH the keyword search and the "
        "citation chain search. Remove duplicates, identify themes, and produce "
        "a thematic synthesis."
    ),
    expected_output="Merged thematic synthesis from both search strategies.",
    agent=summarizer,
    context=[keyword_search_task, citation_search_task],  # Explicit fan-in
)

# Gap finder sees only the synthesis (not raw searches)
focused_gap_task = Task(
    description="Analyze the synthesis for research gaps across 5 dimensions.",
    expected_output="Gap analysis.",
    agent=gap_finder,
    context=[merged_synthesis_task],  # Only sees synthesis, not raw searches
)

parallel_tasks = [keyword_search_task, citation_search_task, merged_synthesis_task, focused_gap_task]

print("PARALLEL DISCOVERY FLOW")
print("=" * 60)
print("  Keyword Search  ──┐")
print("                    ├──> Merged Synthesis ──> Gap Analysis")
print("  Citation Chain  ──┘")
print()
for i, task in enumerate(parallel_tasks, 1):
    ctx = [t.agent.role.split("&")[0].strip() for t in (task.context or [])]
    ctx_str = " + ".join(ctx) if ctx else "none (auto)"
    print(f"  Task {i}: {task.agent.role.split('&')[0].strip()}")
    print(f"    Context from: {ctx_str}")

In [ ]:
# Run the parallel discovery crew
crew_parallel = Crew(
    agents=agents,
    tasks=parallel_tasks,
    process=Process.sequential,  # Still sequential execution, but context controls data flow
    verbose=True,
    memory=False,
)

print(f"Launching parallel-discovery crew — {datetime.now().strftime('%H:%M:%S')}")
result_parallel = crew_parallel.kickoff()
print(f"\nParallel-discovery crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Show parallel flow results
print("PARALLEL DISCOVERY RESULTS")
print("=" * 60)
par_names = ["Keyword Search", "Citation Chain", "Merged Synthesis", "Gap Analysis"]
for name, task in zip(par_names, parallel_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx = [t.agent.role.split("&")[0].strip() for t in (task.context or [])]
    ctx_str = ", ".join(ctx) if ctx else "auto"
    print(f"  {name:<22} {length:>6,} chars | Context: {ctx_str}")

## 12. Key Takeaways

### What We Built
- A **4-agent literature review pipeline** (Searcher → Summarizer → Gap Finder → Bibliography)
- Ran it on **two research topics** (RAG systems + LLM Interpretability)
- Demonstrated **parallel discovery** with explicit `context` for fan-in synthesis

### Multi-Agent Research Flow Concepts
1. **Build-Criticize-Document** — Alternate between constructive (search, synthesize) and critical (gap analysis) phases
2. **Context accumulation** — Later agents see all prior outputs, enabling richer analysis
3. **Cognitive mode separation** — Discovery (divergent) vs. synthesis (convergent) vs. critique (analytical) require different agents
4. **Explicit context** — The `context` parameter enables non-linear flows like parallel discovery

### Agent Design Lessons for Research
- **Searcher:** Emphasize systematic strategies (keyword expansion, citation chaining) not just "find papers"
- **Summarizer:** Force thematic organization, not paper-by-paper summaries — this is the single most impactful design choice
- **Gap Finder:** Use a dedicated critic agent separate from the summarizer to avoid self-validating outputs
- **Bibliography Agent:** Specify citation format (APA 7th) in the backstory to ensure consistency

### Production Tips
- Add web search tools to the Literature Searcher for real paper discovery (Semantic Scholar API, Google Scholar)
- Use `output_pydantic` to enforce structured paper metadata schemas
- Enable `memory=True` for ongoing research programs that accumulate knowledge across reviews
- Consider adding a **Methodology Matcher** agent that maps papers to their research methods
- Add a **Critique Agent** that evaluates the quality of each source before it enters the synthesis